# Session 5 — Tools: controlled capabilities

**Goal:** give an assistant bounded, read-only capabilities and prove the boundaries with checks. *Thread: loop engineering.*

In [ ]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
    print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    LIVE = preflight(REPO_ROOT)  # the configured lane's client; FakeLLM whenever the lane is down

In [ ]:
from bootcamp_agent.checks import check, review

## 1. The tool registry: read the contracts

A tool is a function with a narrow contract the model may call. The registry lists what exists and what each one promises.

In [ ]:
from bootcamp_agent.documents import load_corpus
from bootcamp_agent.llm import FakeLLM
from bootcamp_agent.tools import build_tools

documents = load_corpus(CORPUS_DIR)
client = FakeLLM(default="A three-sentence summary would appear here.")
tools = build_tools(documents, client)
for tool in tools.values():
    print(f"{tool.name:24} {tool.description}")

## 2. Boundaries in action: caps and helpful errors

`max_results=999` is clamped by the tool, never trusted from the caller. An unknown id gets an error that names the valid ones.

In [ ]:
from bootcamp_agent.tools import ToolError

print(tools["search_documents"].run(query="prompt injection defenses", max_results=999))
print()
try:
    tools["get_document_metadata"].run(doc_id="totally-made-up")
except ToolError as error:
    print(f"ToolError: {error}")

## 3. Exercise: a fourth tool with a real contract

**Context.** Four clauses make a contract: what a call returns, what a filtered call returns, and two refusals with helpful messages.

**Instructions.**

1. Clause 1 is done: no tag lists every `doc_id`, one per line.
2. Clause 2: with a tag, only the documents carrying it.
3. Clause 3: an unknown tag raises `ToolError` naming the valid tags.
4. Clause 4: an empty-string tag raises `ToolError`. Validate at the boundary. Then run the check.

In [ ]:
def list_documents(tag: str | None = None) -> str:
    all_tags = {t for doc in documents for t in doc.tags}
    if tag is None:
        return "\n".join(doc.doc_id for doc in documents)  # clause 1, done
    # TODO(you): clause 4, empty-string tag -> ToolError
    # TODO(you): clause 3, unknown tag -> ToolError naming sorted(all_tags)
    # TODO(you): clause 2, only the documents carrying the tag
    raise NotImplementedError


print(list_documents())

**Expected output** (yours may differ in wording, not in shape):

```
agent-loops
evaluation-basics
mcp-overview
prompt-injection
rag-basics
structured-outputs
✅ ch05-e1 passed
```

In [ ]:
check("ch05-e1", list_documents)

## 4. A tool that reaches the outside world

The five-step loop: tool definitions, the model asks for a call with arguments, your code executes it, the result goes back, the model answers. Step 3 is yours, and it is where the boundary lives. `fetch_rates` reaches a free, keyless API (frankfurter.dev). Nothing here spends or mutates.

In [ ]:
import json
import urllib.request


def fetch_rates(base: str) -> dict[str, float]:
    """Live rates for `base` from api.frankfurter.dev. Raises ToolError when offline."""
    url = f"https://api.frankfurter.dev/v1/latest?base={base}"
    try:
        with urllib.request.urlopen(url, timeout=5) as response:
            return json.loads(response.read())["rates"]
    except Exception as error:  # noqa: BLE001 - offline, DNS, 4xx: all are "no rates"
        raise ToolError(f"fetch_rates: could not reach frankfurter.dev ({type(error).__name__})") from error


try:
    print(fetch_rates("USD")["EUR"])
except ToolError as error:
    print(f"skipped the live call: {error}")

## 5. Exercise: convert_currency, validated before it fetches

**Context.** A model will call this tool with whatever arguments it guesses. Every bad argument must be refused *before* any network call happens.

**Instructions.**

1. The happy path is done: `amount` times the rate, formatted with two decimals.
2. Refuse `amount <= 0` with `ToolError`.
3. Refuse a currency code that is not three uppercase letters with `ToolError`.
4. Refuse a target the rates do not contain, naming the known targets. Then run the check: it injects an offline `fetch`, so it runs without network.

In [ ]:
def convert_currency(amount: float, source: str, target: str, fetch=fetch_rates) -> str:
    # TODO(you): amount must be positive
    # TODO(you): source and target must be 3 uppercase letters
    rates = fetch(source)
    # TODO(you): target must be in rates; name sorted(rates) in the error
    converted = amount * rates[target]  # happy path, done
    return f"{amount} {source} = {converted:.2f} {target} (rate {rates[target]})"


print(convert_currency(100, "USD", "EUR", fetch=lambda base: {"EUR": 0.5}))

**Expected output** (yours may differ in wording, not in shape):

```
100 USD = 50.00 EUR (rate 0.5)
✅ ch05-e2 passed
```

In [ ]:
check("ch05-e2", convert_currency)

## 6. Exercise: budgets: the loop that cannot run away

**Context.** `answer_question` takes `max_tool_calls`. With our corpus the direct path rarely needs tools; the point is that the *bound exists and is visible in the trace*.

**Instructions.**

1. Budget 3 is done. Add budget 1 with the same question.
2. Read both traces. Count the `tool_call` events against the budget.
3. Run the check: it confirms neither trace exceeds its budget.

In [ ]:
from bootcamp_agent.agent import answer_question

question = "What defenses help against prompt injection?"
traces = {
    3: [e.kind for e in answer_question(question, documents, FakeLLM(), max_tool_calls=3).trace],  # done
    1: [],  # TODO(you): the same question with max_tool_calls=1
}
for budget, kinds in traces.items():
    print(f"budget={budget}: {kinds}")

**Expected output** (yours may differ in wording, not in shape):

```
budget=3: ['retrieve', 'llm_call', 'decision']
budget=1: ['retrieve', 'llm_call', 'decision']
✅ ch05-e3 passed
```

In [ ]:
check("ch05-e3", traces)

## Exit ticket

Homework: write down one example where the assistant should NOT use a tool, and one where it should ask for human confirmation first. Read `docs/guides/loop-engineering.md`.

## Review

The scorecard for this notebook. Every ❌ line names the exercise and the hint.

In [ ]:
review("ch05")